# Pipeline sencillo Olist - Ingeniería de Datos

Este notebook implementa un pipeline básico de datos para el análisis de comercio electrónico con el dataset de Olist.


In [ ]:
# PARCIAL FINAL - INGENIERÍA DE DATOS
# Pipeline sencillo para analizar comercio electrónico con Olist
# Autor: Johan Sebastian Fuentes Pinto

In [ ]:
# 1. LIBRERÍAS
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Carpeta donde están los archivos CSV
RUTA_DATOS = Path(".")

# Carpeta donde se guardarán resultados
RUTA_SALIDA = Path("resultados_pipeline")
RUTA_SALIDA.mkdir(exist_ok=True)

In [ ]:
# 2. INGESTA DE DATOS

In [ ]:
clientes = pd.read_csv(RUTA_DATOS / "olist_customers_dataset.csv")
geolocalizacion = pd.read_csv(RUTA_DATOS / "olist_geolocation_dataset.csv")
items = pd.read_csv(RUTA_DATOS / "olist_order_items_dataset.csv")
pagos = pd.read_csv(RUTA_DATOS / "olist_order_payments_dataset.csv")
resenas = pd.read_csv(RUTA_DATOS / "olist_order_reviews_dataset.csv")
ordenes = pd.read_csv(RUTA_DATOS / "olist_orders_dataset.csv")
productos = pd.read_csv(RUTA_DATOS / "olist_products_dataset.csv")
vendedores = pd.read_csv(RUTA_DATOS / "olist_sellers_dataset.csv")
traduccion = pd.read_csv(RUTA_DATOS / "product_category_name_translation.csv")

print("Datos cargados correctamente.")

In [ ]:
# 3. EXPLORACIÓN INICIAL

In [ ]:
tablas = {
    "clientes": clientes,
    "geolocalizacion": geolocalizacion,
    "items": items,
    "pagos": pagos,
    "resenas": resenas,
    "ordenes": ordenes,
    "productos": productos,
    "vendedores": vendedores,
    "traduccion": traduccion
}

print("\n--- ESTRUCTURA DE LAS TABLAS ---")
for nombre, tabla in tablas.items():
    print(f"\nTabla: {nombre}")
    print("Filas y columnas:", tabla.shape)
    print("Columnas:", list(tabla.columns))
    print("Duplicados:", tabla.duplicated().sum())
    print("Valores nulos principales:")
    print(tabla.isnull().sum()[tabla.isnull().sum() > 0])

In [ ]:
# 4. LIMPIEZA DE DATOS

In [ ]:
# Eliminar duplicados exactos
for nombre in tablas:
    tablas[nombre] = tablas[nombre].drop_duplicates()

clientes = tablas["clientes"]
geolocalizacion = tablas["geolocalizacion"]
items = tablas["items"]
pagos = tablas["pagos"]
resenas = tablas["resenas"]
ordenes = tablas["ordenes"]
productos = tablas["productos"]
vendedores = tablas["vendedores"]
traduccion = tablas["traduccion"]

# Convertir columnas de fecha a formato datetime
columnas_fecha_ordenes = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for columna in columnas_fecha_ordenes:
    ordenes[columna] = pd.to_datetime(ordenes[columna], errors="coerce")

items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], errors="coerce")
resenas["review_creation_date"] = pd.to_datetime(resenas["review_creation_date"], errors="coerce")
resenas["review_answer_timestamp"] = pd.to_datetime(resenas["review_answer_timestamp"], errors="coerce")

# Traducir categorías de productos de portugués a inglés
productos = productos.merge(traduccion, on="product_category_name", how="left")
productos["product_category_name_english"] = productos["product_category_name_english"].fillna("sin_categoria")

# Agrupar pagos por pedido para no duplicar información al unir tablas
pagos_agrupados = pagos.groupby("order_id", as_index=False).agg({
    "payment_value": "sum",
    "payment_installments": "max",
    "payment_type": "first"
})

pagos_agrupados = pagos_agrupados.rename(columns={
    "payment_value": "valor_pagado_total",
    "payment_installments": "cuotas_maximas",
    "payment_type": "tipo_pago"
})

# Agrupar reseñas por pedido
resenas_agrupadas = resenas.groupby("order_id", as_index=False).agg({
    "review_score": "mean",
    "review_id": "count"
})

resenas_agrupadas = resenas_agrupadas.rename(columns={
    "review_score": "calificacion_promedio",
    "review_id": "cantidad_resenas"
})

print("\nLimpieza terminada.")

In [ ]:
# 5. TRANSFORMACIÓN E INTEGRACIÓN DE DATOS

In [ ]:
# Unir órdenes con clientes
datos = ordenes.merge(clientes, on="customer_id", how="left")

# Unir con items del pedido
datos = datos.merge(items, on="order_id", how="left")

# Unir con productos
datos = datos.merge(
    productos[[
        "product_id",
        "product_category_name_english",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]],
    on="product_id",
    how="left"
)

# Unir con vendedores
datos = datos.merge(
    vendedores[["seller_id", "seller_city", "seller_state"]],
    on="seller_id",
    how="left"
)

# Unir con pagos y reseñas agrupadas
datos = datos.merge(pagos_agrupados, on="order_id", how="left")
datos = datos.merge(resenas_agrupadas, on="order_id", how="left")

# Crear variables útiles para el análisis
datos["valor_item"] = datos["price"] + datos["freight_value"]
datos["fecha_compra"] = datos["order_purchase_timestamp"].dt.date
datos["mes_compra"] = datos["order_purchase_timestamp"].dt.to_period("M").astype(str)

datos["dias_entrega"] = (
    datos["order_delivered_customer_date"] - datos["order_purchase_timestamp"]
).dt.days

datos["dias_retraso"] = (
    datos["order_delivered_customer_date"] - datos["order_estimated_delivery_date"]
).dt.days

datos["pedido_tarde"] = datos["dias_retraso"] > 0

datos["volumen_producto_cm3"] = (
    datos["product_length_cm"] * datos["product_height_cm"] * datos["product_width_cm"]
)

datos["categoria_producto"] = datos["product_category_name_english"].fillna("sin_categoria")

print("\nTransformación terminada.")
print("Tamaño de la tabla final:", datos.shape)

In [ ]:
# 6. ALMACENAMIENTO

In [ ]:
# Guardar dataset procesado en CSV
datos.to_csv(RUTA_SALIDA / "datos_olist_procesados.csv", index=False)

# Intentar guardar también en Parquet, que es más eficiente
try:
    datos.to_parquet(RUTA_SALIDA / "datos_olist_procesados.parquet", index=False)
    print("Datos guardados en CSV y Parquet.")
except Exception:
    print("Datos guardados en CSV. No se pudo guardar en Parquet porque falta pyarrow o fastparquet.")

In [ ]:
# 7. ANÁLISIS EXPLORATORIO

In [ ]:
# 7.1 Ventas por mes
ventas_por_mes = datos.groupby("mes_compra", as_index=False)["valor_item"].sum()
ventas_por_mes = ventas_por_mes.sort_values("mes_compra")

# 7.2 Ventas por estado del cliente
ventas_por_estado = datos.groupby("customer_state", as_index=False)["valor_item"].sum()
ventas_por_estado = ventas_por_estado.sort_values("valor_item", ascending=False)

# 7.3 Categorías con más ingresos
ingresos_por_categoria = datos.groupby("categoria_producto", as_index=False)["valor_item"].sum()
ingresos_por_categoria = ingresos_por_categoria.sort_values("valor_item", ascending=False).head(10)

# 7.4 Tipos de pago más utilizados
pagos_por_tipo = pagos.groupby("payment_type", as_index=False)["payment_value"].sum()
pagos_por_tipo = pagos_por_tipo.sort_values("payment_value", ascending=False)

# 7.5 Relación entre retraso y calificación
calificacion_retraso = datos.groupby("pedido_tarde", as_index=False)["calificacion_promedio"].mean()

print("\n--- RESULTADOS PRINCIPALES ---")
print("\nVentas por mes:")
print(ventas_por_mes.head())

print("\nEstados con mayores ventas:")
print(ventas_por_estado.head(10))

print("\nCategorías con mayores ingresos:")
print(ingresos_por_categoria)

print("\nTipos de pago:")
print(pagos_por_tipo)

print("\nCalificación promedio según retraso:")
print(calificacion_retraso)

In [ ]:
# 8. VISUALIZACIÓN

In [ ]:
# Gráfica 1: ventas por mes
plt.figure(figsize=(12, 5))
plt.plot(ventas_por_mes["mes_compra"], ventas_por_mes["valor_item"], marker="o")
plt.title("Ventas por mes")
plt.xlabel("Mes")
plt.ylabel("Valor vendido")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig(RUTA_SALIDA / "grafica_ventas_por_mes.png")
plt.show()

# Gráfica 2: ventas por estado
plt.figure(figsize=(10, 5))
plt.bar(ventas_por_estado.head(10)["customer_state"], ventas_por_estado.head(10)["valor_item"])
plt.title("Top 10 estados con mayores ventas")
plt.xlabel("Estado del cliente")
plt.ylabel("Valor vendido")
plt.tight_layout()
plt.savefig(RUTA_SALIDA / "grafica_ventas_por_estado.png")
plt.show()

# Gráfica 3: categorías con mayores ingresos
plt.figure(figsize=(12, 5))
plt.bar(ingresos_por_categoria["categoria_producto"], ingresos_por_categoria["valor_item"])
plt.title("Top 10 categorías con mayores ingresos")
plt.xlabel("Categoría")
plt.ylabel("Valor vendido")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig(RUTA_SALIDA / "grafica_categorias_ingresos.png")
plt.show()

# Gráfica 4: tipos de pago
plt.figure(figsize=(8, 5))
plt.bar(pagos_por_tipo["payment_type"], pagos_por_tipo["payment_value"])
plt.title("Valor pagado por tipo de pago")
plt.xlabel("Tipo de pago")
plt.ylabel("Valor pagado")
plt.tight_layout()
plt.savefig(RUTA_SALIDA / "grafica_tipos_pago.png")
plt.show()

# Gráfica 5: calificación según retraso
plt.figure(figsize=(7, 5))
plt.bar(calificacion_retraso["pedido_tarde"].astype(str), calificacion_retraso["calificacion_promedio"])
plt.title("Calificación promedio según entrega tardía")
plt.xlabel("¿Pedido entregado tarde?")
plt.ylabel("Calificación promedio")
plt.tight_layout()
plt.savefig(RUTA_SALIDA / "grafica_calificacion_retraso.png")
plt.show()

In [ ]:
# 9. CONCLUSIONES

In [ ]:
estado_top = ventas_por_estado.iloc[0]["customer_state"]
categoria_top = ingresos_por_categoria.iloc[0]["categoria_producto"]
pago_top = pagos_por_tipo.iloc[0]["payment_type"]

print("\n--- CONCLUSIONES ---")
print(f"1. El estado con mayor valor de ventas es {estado_top}, lo que indica una concentración importante de clientes en esa región.")
print(f"2. La categoría con mayores ingresos es {categoria_top}, por lo tanto puede considerarse una línea clave del negocio.")
print(f"3. El método de pago con mayor valor procesado es {pago_top}, lo que permite entender las preferencias de pago de los clientes.")
print("4. Los pedidos entregados tarde tienden a relacionarse con una menor calificación del cliente.")
print("5. El pipeline permitió integrar, limpiar, transformar, almacenar y analizar los datos de comercio electrónico de forma organizada.")

print("\nPipeline finalizado correctamente.")